# SMART Constrained Variant Training (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/wosac-sim-agents-experiments/blob/main/experiments/smart-constrained/notebooks/smart-constrained_colab.ipynb)

## Objective
- Train constrained SMART variants and persist checkpoints to Drive.
- Keep this notebook training-only.
- Run simulation/evaluation/comparison in dedicated notebooks.


In [ ]:
# Step 1: Sync this repo and bootstrap deterministic Colab runtime
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config
runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch="main", repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


In [ ]:
# Step 2: Load smart-constrained config and define variant checkpoint root
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

from src.workflows import bootstrap_experiment_pack, load_experiment_config

EXPERIMENT_SLUG = "smart-constrained"
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root=".")
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root=".")
display(bundle.summary_table)

RUN = dict(cfg.get("run", {}))
SMART = dict(cfg.get("smart", {}))
CONSTRAINTS = dict(cfg.get("constraints", {}))
SWEEP = dict(cfg.get("sweep", {}))

RUN_NAME = str(RUN.get("run_name", "dev"))
RUN_PREFIX = str(RUN.get("run_prefix", "smart_constrained"))
PERSIST_ROOT = str(RUN.get("persist_root", "/content/drive/MyDrive/wosac_experiments"))
RESUME_FROM_EXISTING = bool(RUN.get("resume_from_existing", True))
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
cfg_hash = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode("utf-8")).hexdigest()
persist_run_dir = Path(PERSIST_ROOT) / f"{RUN_PREFIX}_{RUN_NAME}"
persist_run_dir.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT_DIR = persist_run_dir / "outputs" / RUN_TAG
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VARIANT_CKPT_ROOT = os.environ.get("SMART_VARIANT_CKPT_ROOT", "").strip() or str(persist_run_dir / "checkpoints" / "variants")
print("RUN_TAG:", RUN_TAG)
print("persist_run_dir:", persist_run_dir)
print("RUN_OUTPUT_DIR:", RUN_OUTPUT_DIR)
print("VARIANT_CKPT_ROOT:", VARIANT_CKPT_ROOT)
print("RESUME_FROM_EXISTING:", RESUME_FROM_EXISTING)
print("cfg_hash:", cfg_hash)


In [ ]:
# Step 3: Build constrained variant training grid (train-only)
from src.workflows import run_smart_constrained_flow

flow_bundle = run_smart_constrained_flow(
    run_tag=RUN_TAG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root=".",
    resume_from_existing=RESUME_FROM_EXISTING,
    variant_metrics_dir="",
    smart_checkpoint_root=VARIANT_CKPT_ROOT,
    max_collision_rate=CONSTRAINTS.get("max_collision_rate"),
    max_offroad_rate=CONSTRAINTS.get("max_offroad_rate"),
    max_tl_violation_rate=CONSTRAINTS.get("max_tl_violation_rate"),
    min_diversity_score=CONSTRAINTS.get("min_diversity_score"),
    temperatures=SWEEP.get("temperatures", [0.8, 1.0, 1.2]),
    top_ks=SWEEP.get("top_ks", [8, 16]),
    constraint_weights=SWEEP.get("constraint_weights", [0.05, 0.1, 0.2]),
    smart_repo_url=str(SMART.get("repo_url", "https://github.com/rainmaker22/SMART.git")),
    smart_repo_branch=str(SMART.get("branch", "main")),
    smart_repo_dir=str(SMART.get("repo_dir", "/content/SMART")),
    smart_train_config=str(SMART.get("train_config", "configs/train/train_scalable.yaml")),
    smart_val_config=str(SMART.get("val_config", "configs/validation/validation_scalable.yaml")),
    smart_ckpt_path="",
    smart_raw_data_root=str(SMART.get("raw_data_root", "/content/SMART/data/waymo/scenario")),
    smart_processed_data_root=str(SMART.get("processed_data_root", "/content/SMART/data/waymo_processed")),
    smart_install_pyg=bool(SMART.get("install_pyg", True)),
    sync_smart_repo=True,
)

print("summary:")
print(json.dumps(flow_bundle.summary, indent=2, sort_keys=True))
print("num_variants:", len(flow_bundle.variants))
print("first_variant:")
print(json.dumps(flow_bundle.variants[0], indent=2, sort_keys=True))


In [ ]:
# Step 4: Optional constrained training execution (no evaluation in this notebook)
RUN_SETUP = False
RUN_PREPROCESS = False
RUN_TRAIN = False
VARIANT_IDS_TO_RUN = []  # [] -> run first variant only

stage_flags = {
    "run_setup": bool(RUN_SETUP),
    "run_preprocess": bool(RUN_PREPROCESS),
    "run_train": bool(RUN_TRAIN),
    "variant_ids_to_run": list(VARIANT_IDS_TO_RUN),
}

variant_map = {v["variant_id"]: v for v in flow_bundle.variants}
if VARIANT_IDS_TO_RUN:
    selected_variants = [variant_map[v] for v in VARIANT_IDS_TO_RUN if v in variant_map]
else:
    selected_variants = [flow_bundle.variants[0]] if flow_bundle.variants else []

for variant in selected_variants:
    print("Running variant:", variant["variant_id"])
    cmds = []
    if RUN_SETUP:
        cmds.append(variant["setup_cmd"])
    if RUN_PREPROCESS:
        cmds.extend([variant["preprocess_train_cmd"], variant["preprocess_val_cmd"]])
    if RUN_TRAIN:
        cmds.append(variant["train_cmd"])

    for i, cmd in enumerate(cmds, start=1):
        print(f"[exec {i}/{len(cmds)}] {cmd}")
        subprocess.run(["bash", "-lc", cmd], check=True)

print("Constrained training execution block done.")


In [ ]:
# Step 5: Write constrained training artifact + run manifest (Drive)
from pathlib import Path

from src.workflows import build_training_run_manifest, write_json

stage_flags = globals().get("stage_flags", {"run_setup": False, "run_preprocess": False, "run_train": False, "variant_ids_to_run": []})
variant_ckpt_scan = {}
for v in flow_bundle.variants:
    ckpt_dir = Path(v.get("checkpoint_dir", "")).expanduser()
    ckpts = sorted([str(p) for p in ckpt_dir.glob("*.ckpt")]) if str(ckpt_dir) else []
    variant_ckpt_scan[v["variant_id"]] = {
        "checkpoint_dir": str(ckpt_dir),
        "checkpoint_files": ckpts,
        "resume_checkpoint_path": str(v.get("resume_checkpoint_path", "")),
        "resume_source": str(v.get("resume_source", "")),
    }

payload = {
    "run_id": "smart_constrained_train_v0",
    "status": str(flow_bundle.summary.get("status", "unknown")),
    "run_tag": RUN_TAG,
    "checkpoint_root": VARIANT_CKPT_ROOT,
    "run_output_dir": str(RUN_OUTPUT_DIR),
    "stage_flags": stage_flags,
    "variants": flow_bundle.variants,
    "variant_ckpt_scan": variant_ckpt_scan,
    "flow_summary": flow_bundle.summary,
    "flow_artifacts": flow_bundle.artifacts,
    "notes": [
        "Training-only notebook. Use smart_rollout_simulation_colab.ipynb, smart_checkpoint_eval_colab.ipynb, and smart_comparative_eval_colab.ipynb for simulation/eval/comparison.",
    ],
    "provenance": {
        "config_hash": cfg_hash,
        "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "experiment_slug": EXPERIMENT_SLUG,
    },
}

drive_path = RUN_OUTPUT_DIR / "smart_constrained_train_v0.json"
drive_path.parent.mkdir(parents=True, exist_ok=True)
drive_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

baseline_summary = dict((flow_bundle.baseline or {}).get("summary", {}) or {})
run_manifest = build_training_run_manifest(
    run_id="smart_constrained_train_v0",
    run_tag=RUN_TAG,
    experiment_slug=EXPERIMENT_SLUG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root=".",
    config_hash=cfg_hash,
    flow_summary=baseline_summary,
    stage_flags=stage_flags,
    checkpoint_dir=VARIANT_CKPT_ROOT,
    resume_checkpoint_path="",
    resume_checkpoint_source="variant_specific",
    extra={
        "flow_artifacts": flow_bundle.artifacts,
        "num_variants": len(flow_bundle.variants),
        "variant_resume_summary": {k: {"resume_checkpoint_path": v.get("resume_checkpoint_path", ""), "resume_source": v.get("resume_source", "")} for k, v in variant_ckpt_scan.items()},
    },
)
manifest_path = RUN_OUTPUT_DIR / "run_manifest.json"
write_json(manifest_path, run_manifest)

print("drive_path:", drive_path)
print("manifest_path:", manifest_path)
print("variants_scanned:", len(variant_ckpt_scan))
